In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_dev")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "storageproject99")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/matricula/"

In [0]:
matricula_schema = StructType(fields=[StructField("ACAD_CAREER", StringType(), True),
                                      StructField("STRM", StringType(), True),
                                      StructField("MODALIDAD", StringType(), True),
                                      StructField("EMPLID", StringType(), True),
                                      StructField("EMPLID_NAME", StringType(), True),
                                      StructField("ACAD_PROG", StringType(), True),
                                      StructField("ACAD_PLAN", StringType(), True),
                                      StructField("COURSE", StringType(), True),
                                      StructField("COURSE_DESC", StringType(), True),
                                      StructField("PROM", IntegerType(), True),
                                      StructField("TURN", StringType(), True),
                                      StructField("TURN_DESC", StringType(), True),
                                      StructField("CAMPUS", StringType(), True),
                                      StructField("CYCLE", StringType(), True),
                                      StructField("NBR_CLASS", StringType(), True),
                                      StructField("SECTION", StringType(), True)
                                      ])

In [0]:
df_matricula_read = spark.read.option('header',True)\
                              .schema(matricula_schema)\
                              .csv(ruta)

In [0]:
df_matricula_ingestion_date = df_matricula_read.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_matricula_final = df_matricula_ingestion_date.select("ACAD_CAREER",
                                                        "STRM",
                                                        "MODALIDAD",
                                                        "EMPLID",
                                                        "EMPLID_NAME",
                                                        "ACAD_PROG",
                                                        "ACAD_PLAN",
                                                        "COURSE",
                                                        "COURSE_DESC",
                                                        "PROM",
                                                        "TURN",
                                                        "TURN_DESC",
                                                        "CAMPUS",
                                                        "CYCLE",
                                                        "NBR_CLASS",
                                                        "SECTION",
                                                        "INGESTION_DATE"
                                                        )

In [0]:
df_matricula_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.matricula")